# AI Education Center Finder — Uzbekistan

This notebook supports the GitHub/Streamlit repository included with the project. The app is restricted to Uzbekistan and uses a local, explainable recommendation model that does not require an OpenAI API key.

> **Data notice:** the bundled center records are synthetic demonstration data. Replace them with verified data before public deployment.

## 1. Open the project in Colab

Upload the downloaded ZIP to the Colab session, then run the next cell. It extracts the repository and changes the working directory to the app folder.

In [ ]:
from google.colab import files
import os
import zipfile
from pathlib import Path

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise ValueError('Upload the Uzbekistan project ZIP file.')

zip_path = Path(zip_names[0])
extract_root = Path('/content/uzbekistan_education_center_app')
extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as archive:
    archive.extractall(extract_root)

# The ZIP may contain either root files or one top-level folder.
candidates = list(extract_root.rglob('app.py'))
if not candidates:
    raise FileNotFoundError('app.py was not found after extraction.')
project_dir = candidates[0].parent
os.chdir(project_dir)
print('Project directory:', project_dir)
print('
'.join(sorted(p.name for p in project_dir.iterdir())))

## 2. Install the Streamlit project dependencies

In [ ]:
!pip -q install -r requirements.txt
print('Dependencies installed.')

## 3. Validate the Uzbekistan-only dataset

The checks below confirm that the CSV contains only Uzbekistan records and covers the expected 14 top-level administrative entries used by the demo.

In [ ]:
import pandas as pd

DATA_FILE = 'uzbekistan_education_centers.csv'
df = pd.read_csv(DATA_FILE)

assert not df.empty, 'The dataset is empty.'
assert set(df['country'].dropna().unique()) == {'Uzbekistan'}, 'Non-Uzbekistan data found.'
assert df['region'].nunique() == 14, 'Expected demonstration coverage of 14 region-level entries.'
assert df[['latitude', 'longitude']].notna().all().all(), 'Missing map coordinates.'

print('Rows:', len(df))
print('Regions:', df['region'].nunique())
print('Cities:', df['city'].nunique())
display(df[['name', 'region', 'city', 'center_type', 'languages']].head(10))

## 4. Recommendation architecture

The app creates a user query from role, learning goals, subjects, resources, preferred center types, and languages. It compares that query with each center profile using TF-IDF character n-grams. The final transparent score combines:

- 43% multilingual content similarity
- 19% grade/age/membership eligibility
- 15% selected-time opening status
- 13% estimated distance
- 10% affordability

Character n-grams provide basic matching for English, Uzbek, and Russian without sending user data to an external AI service.

## 5. Import and smoke-test the app functions

In [ ]:
import datetime as dt
import app

centers = app.load_centers()
assert len(centers) > 0

sample = centers.iloc[0]
distance = app.haversine_km(41.3111, 69.2797, sample['latitude'], sample['longitude'])
minutes = app.estimate_commute_minutes(distance, 'Public transport')

print('Loaded centers:', len(centers))
print('First center:', sample['name'])
print('Example distance from Tashkent:', round(distance, 1), 'km')
print('Simple commute estimate:', minutes, 'minutes')

## 6. Run the app

For the most reliable public deployment, push the repository to GitHub and connect it to Streamlit Community Cloud with `app.py` as the main file.

To run from a terminal on your own computer:

```bash
streamlit run app.py
```

Colab does not directly expose Streamlit ports publicly without an additional tunnel service. The repository intentionally does not require or store tunnel tokens.

## 7. Package a clean ZIP from Colab

In [ ]:
from pathlib import Path
import shutil

project_dir = Path.cwd()
zip_base = Path('/content/uzbekistan_education_center_app_github_streamlit')
zip_file = shutil.make_archive(str(zip_base), 'zip', root_dir=project_dir)
print('Created:', zip_file)

# Uncomment to download from Colab:
# from google.colab import files
# files.download(zip_file)

## 8. Production checklist

- Replace all `[Demo]` records with verified education-center data.
- Confirm addresses, coordinates, hours, fees, languages, and admissions requirements.
- Obtain permission before publishing non-public contact information.
- Add a center-owner correction and verification workflow.
- Avoid storing private student data in GitHub.
- Treat AI ranking as decision support rather than a quality guarantee.
- Add database storage and authenticated accounts only after privacy and security design.